# 📡 GuardDuty tool quickstart: give your agent a live security feed

This notebook walks through the one new tool added to **SOC It To 'Em**: `get_guardduty_findings`
([`lambda/guardduty_tool.py`](../lambda/guardduty_tool.py)). It reads **Amazon GuardDuty** findings and turns each
one into short, redacted facts an LLM agent can reason about.

**It runs without AWS**: the cells below use a GuardDuty-shaped sample finding and a stand-in client. To try it
against your own account, set `LIVE = True` in step 5 (you need AWS credentials and GuardDuty enabled; the tool
only reads, it never changes anything).

Demo video: https://www.youtube.com/watch?v=u9MZofCVn0s

In [1]:
import json, os, sys
sys.path.insert(0, os.path.abspath(os.path.join('..', 'lambda')))  # run from the notebooks/ folder
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

import guardduty_tool as gd
print(json.dumps(gd.TOOL, indent=2))

{
  "name": "get_guardduty_findings",
  "description": "Read live security findings from Amazon GuardDuty in this AWS account. With no finding_id, list the most recent active findings. With a finding_id (GD-...), get that finding's facts: what happened, the affected resource, the remote IP and where it is, how often it was seen, and GuardDuty's severity. Account IDs are redacted.",
  "schema": {
    "type": "object",
    "properties": {
      "finding_id": {
        "type": "string",
        "description": "A GD-... id, or omit to list."
      },
      "max_results": {
        "type": "integer",
        "description": "How many to list (1-10)."
      }
    }
  }
}


## 1. A finding, as the GuardDuty API returns it

Shaped like GuardDuty's own *sample* findings (made-up values; the account ID is a placeholder).

In [2]:
ACCOUNT = '123456789012'
FINDING = {
    'AccountId': ACCOUNT, 'Arn': f'arn:aws:guardduty:us-east-1:{ACCOUNT}:detector/d1/finding/a1b2c3d4',
    'Id': 'a1b2c3d4e5f60718293a4b5c6d7e8f90', 'Region': 'us-east-1', 'Severity': 8,
    'Type': 'UnauthorizedAccess:EC2/SSHBruteForce', 'UpdatedAt': '2026-10-24T15:02:00.000Z',
    'Title': '198.51.100.7 is performing SSH brute force attacks against i-0abc12345.',
    'Description': '198.51.100.7 is performing SSH brute force attacks against i-0abc12345.',
    'Resource': {'ResourceType': 'Instance',
                 'InstanceDetails': {'InstanceId': 'i-0abc12345', 'InstanceType': 't3.micro',
                                     'NetworkInterfaces': [{'PublicIp': '203.0.113.10'}],
                                     'Tags': [{'Key': 'Name', 'Value': 'web-01'}]},
                 'AccessKeyDetails': {'AccessKeyId': 'AKIAABCDEFGHIJKL1234', 'UserName': 'deploy-bot'}},
    'Service': {'Count': 41, 'EventFirstSeen': '2026-10-24T14:40:00.000Z', 'EventLastSeen': '2026-10-24T15:01:00.000Z',
                'AdditionalInfo': {'Value': '{"sample":true}', 'Type': 'default'},
                'Action': {'ActionType': 'NETWORK_CONNECTION', 'NetworkConnectionAction': {
                    'ConnectionDirection': 'INBOUND', 'Protocol': 'TCP', 'Blocked': False,
                    'LocalPortDetails': {'Port': 22, 'PortName': 'SSH'},
                    'RemoteIpDetails': {'IpAddressV4': '198.51.100.7', 'Country': {'CountryName': 'Exampleland'},
                                        'Organization': {'AsnOrg': 'Example Hosting Ltd'}}}},
                'Evidence': {'ThreatIntelligenceDetails': [{'ThreatListName': 'ExampleList', 'ThreatNames': ['ssh-scanner']}]}},
}
print(len(json.dumps(FINDING)), 'characters of raw finding')

1403 characters of raw finding


## 2. What the model actually sees: `finding_summary()`

Only the facts that matter for triage, with the **account ID and access key redacted**.

In [3]:
summary = gd.finding_summary(FINDING)
print(json.dumps(summary, indent=2, ensure_ascii=False))
assert ACCOUNT not in json.dumps(summary) and 'AKIAABCDEFGHIJKL1234' not in json.dumps(summary)

{
  "alert_id": "GD-a1b2c3d4e5f60718293a4b5c6d7e8f90",
  "source": "Amazon GuardDuty",
  "finding_type": "UnauthorizedAccess:EC2/SSHBruteForce",
  "title": "198.51.100.7 is performing SSH brute force attacks against i-0abc12345.",
  "description": "198.51.100.7 is performing SSH brute force attacks against i-0abc12345.",
  "severity": "high (8 of 10)",
  "region": "us-east-1",
  "first_seen": "2026-10-24T14:40:00.000Z",
  "last_seen": "2026-10-24T15:01:00.000Z",
  "times_seen": 41,
  "activity": "NETWORK_CONNECTION",
  "sample_finding": true,
  "resource": {
    "type": "Instance",
    "instance_id": "i-0abc12345",
    "instance_type": "t3.micro",
    "public_ip": "203.0.113.10",
    "tags": [
      "Name=web-01"
    ],
    "user": "deploy-bot",
    "access_key": "AKIA…1234"
  },
  "remote_ip": {
    "ip": "198.51.100.7",
    "country": "Exampleland",
    "city": null,
    "network_owner": "Example Hosting Ltd"
  },
  "connection": {
    "direction": "INBOUND",
    "protocol": "TCP",
 

## 3. Same function, EventBridge shape

EventBridge delivers the finding as the event's `detail`, with camelCase keys. The summary comes out identical,
so one function serves both "the agent asks" and "GuardDuty rings the bell".

In [4]:
def camel(o):
    if isinstance(o, dict):
        return {k[:1].lower() + k[1:]: camel(v) for k, v in o.items()}
    return [camel(v) for v in o] if isinstance(o, list) else o

event = {'source': 'aws.guardduty', 'detail-type': 'GuardDuty Finding', 'detail': camel(FINDING)}
print(gd.finding_summary(event['detail']) == summary)

True


## 4. The tool call itself

With no id it lists recent active findings (short lists carry the full facts); with a `GD-…` id it returns
one finding. Errors come back as messages the model can repeat, never as guesses.

In [5]:
class StandInGuardDuty:
    """Answers like the boto3 GuardDuty client, from the sample finding above."""
    def list_detectors(self): return {'DetectorIds': ['d1']}
    def list_findings(self, **kw): return {'FindingIds': [FINDING['Id']]}
    def get_findings(self, DetectorId, FindingIds): return {'Findings': [FINDING]}

gd._client, gd._detector = StandInGuardDuty(), None
print(json.dumps(gd.get_guardduty_findings({'max_results': 1}), indent=2)[:600], '...')
print(gd.get_guardduty_findings({'finding_id': 'GD-' + FINDING['Id']})['severity'])

{
  "findings": [
    {
      "alert_id": "GD-a1b2c3d4e5f60718293a4b5c6d7e8f90",
      "source": "Amazon GuardDuty",
      "finding_type": "UnauthorizedAccess:EC2/SSHBruteForce",
      "title": "198.51.100.7 is performing SSH brute force attacks against i-0abc12345.",
      "description": "198.51.100.7 is performing SSH brute force attacks against i-0abc12345.",
      "severity": "high (8 of 10)",
      "region": "us-east-1",
      "first_seen": "2026-10-24T14:40:00.000Z",
      "last_seen": "2026-10-24T15:01:00.000Z",
      "times_seen": 41,
      "activity": "NETWORK_CONNECTION",
      "samp ...
high (8 of 10)


## 5. Optional: your own account

Set `LIVE = True` to read real findings (needs credentials with `guardduty:ListDetectors`, `ListFindings`,
`GetFindings`; see [`docs/iam-guardduty-policy.json`](../docs/iam-guardduty-policy.json)).

In [6]:
LIVE = False
if LIVE:
    gd._client, gd._detector = None, None  # back to the real boto3 client
    print(json.dumps(gd.get_guardduty_findings({'max_results': 3}), indent=2))
else:
    print('Skipped: set LIVE = True to read your own GuardDuty findings.')

Skipped: set LIVE = True to read your own GuardDuty findings.


## 6. Plug it into your agent

**Amazon Bedrock Converse API** (how SOC It To 'Em does it):

```python
tool_config = {"tools": [{"toolSpec": {"name": gd.TOOL["name"], "description": gd.TOOL["description"],
                                       "inputSchema": {"json": gd.TOOL["schema"]}}}]}
resp = bedrock.converse(modelId="us.amazon.nova-lite-v1:0", messages=messages, toolConfig=tool_config)
# when resp["stopReason"] == "tool_use": result = gd.get_guardduty_findings(tool_use["input"])
```

**Strands Agents:**

```python
from strands import Agent, tool

@tool
def get_guardduty_findings(finding_id: str = "", max_results: int = 5) -> dict:
    """Read live Amazon GuardDuty findings: list recent ones, or get one by GD- id."""
    return gd.get_guardduty_findings({"finding_id": finding_id, "max_results": max_results})

agent = Agent(tools=[get_guardduty_findings])
```

**Make it live**: an EventBridge rule with the pattern below, targeting your agent's Lambda, wakes it for every new
finding (setup steps in [`docs/GUARDDUTY.md`](../docs/GUARDDUTY.md)):

```json
{ "source": ["aws.guardduty"], "detail-type": ["GuardDuty Finding"] }
```